##### Step 1: Convert CSV to JSON Files

In [2]:
import csv
import json
import uuid
import os

raw_reviews_file = "../data/raw/hotel_reviews_1000.csv"
transformed_dir = "../data/transformed"

raw_reviews = open(raw_reviews_file, "r").readlines()

if not os.path.exists(transformed_dir):
    os.makedirs(transformed_dir)

def process_reviews(file_path):
    with open(file_path, 'r', newline='', encoding='utf-8') as csvfile:
        # Read the first line to get the header
        header = next(csv.reader(csvfile))
        
        # Create a mapping of expected column names to actual column names
        column_mapping = {
            'dateAdded': 'dateAdded',
            'city': 'city',
            'hotel_name': 'name',
            'hotel_state': 'province',
            'review_text': 'reviews.text',
            'review_title': 'reviews.title'
        }
        
        # Find the index of each required column
        column_indices = {}
        for expected_name, actual_name in column_mapping.items():
            try:
                column_indices[expected_name] = header.index(actual_name)
            except ValueError:
                print(f"Warning: Column '{actual_name}' not found in the CSV. Some data may be missing.")
        
        # Reset file pointer to the beginning
        csvfile.seek(0)
        
        # Skip the header row
        next(csvfile)
        
        # Use csv.reader instead of DictReader
        reader = csv.reader(csvfile)
        
        for i, row in enumerate(reader, start=1):
            review_json = {}
            for key, index in column_indices.items():
                if index < len(row):
                    review_json[key] = row[index]
                else:
                    review_json[key] = ""  # or None, depending on your preference
            
            # Generate a unique identifier
            review_json['id'] = str(uuid.uuid4())
            
            # print(json.dumps(review_json, indent=2))
            print(f"processed record [{i}] with id [{review_json['id']}]")

            with open(f"{transformed_dir}/review_{i}.json", "w+") as f:
                json.dump(review_json, f, indent=2)
            
process_reviews(raw_reviews_file)

processed record [1] with id [61fd8b40-18c2-4f10-8c7d-a75280e6749b]
processed record [2] with id [3bd4b1c6-b181-4011-bfc1-b65395663ed0]
processed record [3] with id [fe7471af-ac74-41df-b4e4-21c35577647a]
processed record [4] with id [8e2e6cde-742c-42ca-b4f6-27d5be0cbf4f]
processed record [5] with id [eff804c6-21e6-4789-b0b8-e890193fd585]
processed record [6] with id [84b7cf9c-ed67-4394-86f6-0615e6ef13d9]
processed record [7] with id [cf0fa5fc-8ab8-4115-9f75-6ed839de419b]
processed record [8] with id [d9187704-8ebb-40c0-b962-49ac43bc9569]
processed record [9] with id [aa4421cf-7972-4ab0-a108-ba333b7d1ebc]
processed record [10] with id [11898b35-1268-4118-bbf0-ddd34e89fe83]
processed record [11] with id [72fe18cc-50b1-4b18-b4bc-7d58fc5033ae]
processed record [12] with id [6c8cbaf3-e955-43ed-800a-1c13ea9db916]
processed record [13] with id [eb915dc6-a7fc-4d87-b3cf-0dde0f0ab550]
processed record [14] with id [e86a6e2d-6ca1-4e97-a652-9095daa4688b]
processed record [15] with id [4be4f802-d64

#### Step 2: Create Embeddings for each of the JSON Files

In [3]:
%pip install -q python-dotenv openai


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

response = client.embeddings.create(
    input="Hello world",
    model="text-embedding-3-small"
)

print(len(response.data[0].embedding))
print(response.data[0].embedding)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: 7nD1S0VD************************************************************************7KjI. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [4]:
import os
import json

transformed_dir = "../data/transformed"
embedded_dir = "../data/embedded"

if not os.path.exists(embedded_dir):
    os.makedirs(embedded_dir)
    
def prepare_embedding_str(review_json):
    return f"REVIEW_TITLE: {review_json['review_title']} REVIEW_TEXT: {review_json['review_text']} HOTEL_NAME: {review_json['hotel_name']} HOTEL_CITY: {review_json['city']} HOTEL_STATE: {review_json['hotel_state']}"
    
client = OpenAI()
for file in os.listdir(transformed_dir):
    with open(f"{transformed_dir}/{file}", "r") as f:
        review = json.load(f)
        
        ## start here
        embedding_str = prepare_embedding_str(review)
        response = client.embeddings.create(
            input=embedding_str,
            model="text-embedding-3-small"
        )
        
        review['embedding'] = response.data[0].embedding
        
        with open(f"{embedded_dir}/{file}", "w") as f:
            json.dump(review, f, indent=2)